<a href="https://colab.research.google.com/github/jcpd2806/data-mining/blob/main/Operacion_Cazadores_Bots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 1: Limpieza y Auditoria


In [ ]:
import pandas as pd
import json as js
import matplotlib as plot

url = "./raw_perfiles_usuarios.csv"
raw_perfiles_usuarios = pd.read_csv(url)
print(raw_perfiles_usuarios.isna().sum())

print("duplicados",raw_perfiles_usuarios.duplicated().sum())


row_id                  0
user_id                 0
fecha_creacion          0
edad_usuario           61
pais                   36
dispositivo             0
nivel_verificacion      0
seguidores             26
siguiendo               0
posts_totales           0
tasa_interaccion       45
hora_pico_actividad     0
version_api             0
dtype: int64
duplicados 0


In [ ]:
display(raw_perfiles_usuarios)

,row_id,user_id,fecha_creacion,edad_usuario,pais,dispositivo,nivel_verificacion,seguidores,siguiendo,posts_totales,tasa_interaccion,hora_pico_actividad,version_api
0,ROW_0001,USR_0704,2023-02-07,65.0,ARGENTINA,IOS,CORREO,434.0,124,13,0.1297,15,v2.4_stable
1,ROW_0002,USR_0901,2026-02-15,27.0,ESPANA,WEB,TELEFONO,40.0,1388,892,0.0133,3,v2.4_stable
2,ROW_0003,USR_0774,2024-04-06,34.0,CHILE,IOS,VERIFICADO,47.0,407,34,0.1938,9,v2.4_stable
3,ROW_0004,USR_0141,2023-09-29,20.0,MEXICO,ANDROID,VERIFICADO,233.0,591,129,0.4488,10,v2.4_stable
4,ROW_0005,USR_9077,2023-03-10,145.0,COLOMBIA,IOS,TELEFONO,300.0,200,45,0.2200,18,v2.4_stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,ROW_1246,USR_9045,2024-05-12,28.0,MEXICO,ANDROID,CORREO,377.0,180,52,0.1500,14,v2.4_stable
1246,ROW_1247,USR_0827,2026-08-19,26.0,ARGENTINA,API_SCRIPT,NINGUNO,118.0,3294,849,0.0404,5,v2.4_stable
1247,ROW_1248,USR_9096,2024-08-20,25.0,ARGENTINA,WEB,CORREO,-150.0,150,80,0.2000,20,v2.4_stable
1248,ROW_1249,USR_0861,2023-07-21,33.0,chile,IOS,TELEFONO,81.0,528,59,0.2789,11,v2.4_stable


# Linette


In [ ]:
#%%
##%%
with open("raw_actividad_logs.json", encoding="utf-8") as f:
  data = json.load(f)


df = pd.json_normalize(data)

df_copy_json = df.copy()
display(df)

df_copy_json.info()
df_copy_json = df_copy_json.dropna(subset=['account_id'])
df_copy_json.info()

#%%
df_copy_json = df_copy_json.dropna(subset=['engagement_stats.avg_session_min'])
df_copy_json = df_copy_json[df_copy_json['engagement_stats.avg_session_min'] >= 0]
df_copy_json = df_copy_json[df_copy_json['engagement_stats.variance_post_interval_hr'] >= 0]
df_copy_json = df_copy_json[df_copy_json['engagement_stats.likes_given_30d'] >= 0]
df_copy_json.info()


df_copy_json['account_id'] = df_copy_json['account_id'].apply(lambda x: str(x).upper().strip())


### retwets
filas = []

# zip() recorre ambas columnas fila por fila al mismo tiempo
for origin, retweets in zip(df['account_id'], df['network_activity.retweets_to']):
  # Verificar que retweets sea una lista válida
  if isinstance(retweets, list):
    for retweet in retweets:
      retweet = retweet.upper()
      if origin != retweet:
        filas.append({
            'Origen': origin,
            'Destino': retweet,
            'Relacion': 'Retweet',
        })

df_rt = pd.DataFrame(filas).drop_duplicates()

#metricas
filas_met = []

for cuenta in data:
  meta = cuenta.get('profile_meta') or {}
  filas_met.append({
      "usuario": cuenta['account_id'].upper(),
      'email': meta.get('email_domain'),
      "ip": meta.get('last_ip'),
      'tarjeta': meta.get('payment_hash'),
  })

df_meta = pd.DataFrame(filas_met).drop_duplicates()

# tarjetas
#%%
df_tarjeta = df_meta.dropna(subset=['tarjeta'])[["usuario", "tarjeta"]].drop_duplicates()
same_card = df_tarjeta.merge(df_tarjeta, on='tarjeta')
same_card = same_card[same_card['usuario_x'] < same_card['usuario_y']]

df_tarjeta = pd.DataFrame({
    'Origen': same_card['usuario_x'],
    'Destino': same_card['usuario_y'],
    'Relacion': 'Misma_tarjeta'
})


# email
#%%
df_email = df_meta.dropna(subset=['email'])[["usuario", "email"]].drop_duplicates()

#%%
same_domain = df_email.merge(df_email, on='email')

#%%
same_domain = same_domain[same_domain['usuario_x'] < same_domain['usuario_y']]

df_email = pd.DataFrame({
    'Origen': same_domain['usuario_x'],
    'Destino': same_domain['usuario_y'],
    'Relacion': 'Mismo Dominio',
})

# ip
#%%
df_ip = df_meta.dropna(subset=['ip'])[["usuario", "ip"]].drop_duplicates()
same_ip = df_ip.merge(df_ip, on='ip')
same_ip = same_ip[same_ip['usuario_x'] < same_ip['usuario_y']]

df_ip = pd.DataFrame({
    'Origen': same_ip['usuario_x'],
    'Destino': same_ip['usuario_y'],
    'Relacion': 'Mismo ip',
})


df_maestra = pd.concat([df_rt, df_tarjeta, df_email, df_ip], ignore_index=True).drop_duplicates().reset_index(drop=True)

,account_id,profile_meta.email_domain,profile_meta.last_ip,profile_meta.payment_hash,engagement_stats.avg_session_min,engagement_stats.variance_post_interval_hr,engagement_stats.likes_given_30d,network_activity.retweets_to,profile_meta,engagement_stats,network_activity
0,USR_0232,tempmail.xyz,104.28.19.102,VISA-4455-XXXX-9012,4.56,0.74,550.0,"[USR_0246, USR_0249, USR_0052, USR_0686, USR_0...",NaN,NaN,NaN
1,USR_0497,outlook.com,189.124.92.199,MC-2783-XXXX-3879,27.37,7.26,306.0,"[USR_0084, USR_0862, USR_0288, USR_0880, USR_0...",NaN,NaN,NaN
2,USR_0772,outlook.com,189.248.246.152,None,17.82,15.02,251.0,[USR_0007],NaN,NaN,NaN
3,USR_0476,tempmail.xyz,104.28.19.108,VISA-4455-XXXX-9013,3.30,0.27,1579.0,"[USR_0539, USR_0226, USR_0980, USR_0189]",NaN,NaN,NaN
4,USR_0425,yahoo.com,189.28.121.228,MC-2963-XXXX-7253,20.59,12.85,214.0,"[USR_0306, USR_0424, USR_0056, USR_0892, USR_0...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1125,USR_0759,outlook.com,189.68.124.96,None,NaN,12.52,118.0,[USR_0152],NaN,NaN,NaN
1126,USR_0884,proton.me,189.195.157.240,None,23.84,8.79,74.0,"[USR_0238, USR_0732, USR_0063, USR_0541]",NaN,NaN,NaN
1127,usr_0540,gmail.com,189.70.189.62,None,9.55,12.87,394.0,"[USR_0235, USR_0208]",NaN,NaN,NaN
1128,USR_0746,gmail.com,189.170.119.181,None,21.59,6.94,45.0,[USR_0395],NaN,NaN,NaN


In [ ]:
display(df_maestra)


,Origen,Destino,Relacion
0,USR_0232,USR_0246,Retweet
1,USR_0232,USR_0249,Retweet
2,USR_0232,USR_0052,Retweet
3,USR_0232,USR_0686,Retweet
4,USR_0232,USR_0927,Retweet
...,...,...,...
174114,USR_0656,USR_0974,Mismo ip
174115,USR_0656,USR_0556,Mismo ip
174116,USR_0656,USR_0852,Mismo ip
174117,USR_0747,USR_0798,Mismo ip


# Eric

In [ ]:
from numpy import e
# Reglas de Clasificación

raw_actividad_logs = pd.read_json("raw_actividad_logs.json")
print('ACTIVIDAD LOGS INFO:')
raw_actividad_logs.info()

raw_perfiles_usuarios_sucio = pd.read_csv("raw_perfiles_usuarios.csv")
raw_perfiles_usuarios = raw_perfiles_usuarios_sucio
print('PERFILES USUARIOS INFO:')
raw_perfiles_usuarios.info()

print('NA EN PERFILES: ', raw_perfiles_usuarios.isna().sum())

# 0 - Registro Válido

# rellenar n/a's
raw_perfiles_usuarios['pais']= raw_perfiles_usuarios['pais'].fillna('desconocido')
raw_perfiles_usuarios['edad_usuario']= raw_perfiles_usuarios['edad_usuario'].fillna(raw_perfiles_usuarios['edad_usuario'].median())
raw_perfiles_usuarios['seguidores']= raw_perfiles_usuarios['seguidores'].fillna(raw_perfiles_usuarios['seguidores'].median())
raw_perfiles_usuarios['tasa_interaccion']= raw_perfiles_usuarios['tasa_interaccion'].fillna(raw_perfiles_usuarios['tasa_interaccion'].median())

# rellenar corruptos o huerfanos
mediana_edad = raw_perfiles_usuarios['edad_usuario'].median()
condicion_edad = (raw_perfiles_usuarios['edad_usuario'] < 13) | (raw_perfiles_usuarios['edad_usuario'] > 100)
raw_perfiles_usuarios.loc[condicion_edad, 'edad_usuario'] = mediana_edad

mediana_seguidores = raw_perfiles_usuarios['seguidores'].median()
condicion_seguidores = (raw_perfiles_usuarios['seguidores'] < 1) | (raw_perfiles_usuarios['seguidores'] > 10000)
raw_perfiles_usuarios.loc[condicion_seguidores, 'seguidores'] = mediana_seguidores

mediana_tasa_interaccion = raw_perfiles_usuarios['tasa_interaccion'].median()
condicion_tasa_interaccion = (raw_perfiles_usuarios['tasa_interaccion'] < 0) | (raw_perfiles_usuarios['tasa_interaccion'] > 1)
raw_perfiles_usuarios.loc[condicion_tasa_interaccion, 'tasa_interaccion'] = mediana_tasa_interaccion

print('NA EN PERFILES: ', raw_perfiles_usuarios.isna().sum())

display(raw_perfiles_usuarios.describe())

# comparar raw_perfiles_usuarios.con raw_perfiles_usuarios_sucio y unicament mostrar row_id
print(raw_perfiles_usuarios['row_id'].compare(raw_perfiles_usuarios_sucio['row_id']))


ACTIVIDAD LOGS INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1130 entries, 0 to 1129
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   account_id        1130 non-null   object
 1   profile_meta      1105 non-null   object
 2   engagement_stats  1105 non-null   object
 3   network_activity  1105 non-null   object
dtypes: object(4)
memory usage: 35.4+ KB
PERFILES USUARIOS INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   row_id               1250 non-null   object 
 1   user_id              1250 non-null   object 
 2   fecha_creacion       1250 non-null   object 
 3   edad_usuario         1189 non-null   float64
 4   pais                 1214 non-null   object 
 5   dispositivo          1250 non-null   object 
 6   nivel_verificacion  

,edad_usuario,seguidores,siguiendo,posts_totales,tasa_interaccion,hora_pico_actividad
count,1250.000000,1250.000000,1250.00000,1250.000000,1250.000000,1250.00000
mean,36.840000,413.845600,606.77520,210.545600,0.235517,13.24240
std,13.389108,581.631898,764.13201,320.290738,0.123240,6.43148
min,16.000000,2.000000,32.00000,2.000000,0.005000,0.00000
25%,26.000000,97.250000,158.25000,43.000000,0.150000,9.00000
50%,34.000000,230.500000,298.50000,84.000000,0.220000,14.00000
75%,47.000000,447.750000,723.25000,252.000000,0.312125,18.00000
max,65.000000,5318.000000,6124.00000,3325.000000,0.675800,23.00000


Empty DataFrame
Columns: [self, other]
Index: []


# JP

# Esteban